In [102]:
import psycopg
from pathlib import Path

In [103]:
PROJECT_ROOT = Path("..")
FINAL_DATA = PROJECT_ROOT / "data" / "final"

print(FINAL_DATA.resolve())


C:\data science\E-Commerce Data Intelligence Platform\data\final


In [104]:
files = sorted(FINAL_DATA.glob("*.csv"))

for file in files:
    print(file.name)

dim_customers.csv
dim_products.csv
dim_sellers.csv
fact_order_items.csv
fact_orders.csv
fact_payments.csv
fact_reviews.csv


In [105]:
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="ecommerce_db",
    user="postgres",
    password="meghana@123"
)

#conn is your database connection.

In [106]:
with conn.cursor() as cur:
    cur.execute("SELECT current_database();")#This sends this SQL query to PostgreSQL
    print(cur.fetchone())#Give me the first row of the result.

#A cursor is what Python uses to send SQL commands to PostgreSQL and retrieve the results.
#The with statement also makes sure the cursor is properly closed when you're finished with it.

('ecommerce_db',)


In [107]:
#Give me the names of all tables in the public schema.

with conn.cursor() as cur:
    cur.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name; 
    """)

    for row in cur.fetchall():
        print(row[0])#Because each row is a tuple.

dim_customers
dim_products
dim_sellers
fact_order_items
fact_orders
fact_payments
fact_reviews


In [108]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM fact_order_items;")#Count every row in fact_order_items.
    print(cur.fetchone()[0])

0


Why we're using COPY

This is an important data-engineering concept.

We have:

112,650 rows

We don't want to do:

for row in rows:
    INSERT ...

That's unnecessarily slow.

PostgreSQL has a bulk-loading mechanism called:

COPY

Conceptually:

CSV
 ↓
COPY
 ↓
PostgreSQL table

This is much more appropriate for loading a large CSV.

In [109]:
import os

print(os.getcwd())

c:\data science\E-Commerce Data Intelligence Platform\notebooks


In [110]:
from pathlib import Path

print(FINAL_DATA)
print(FINAL_DATA.exists())

..\data\final
True


In [111]:
import pandas as pd

products_check = pd.read_csv(FINAL_DATA / "dim_products.csv")

print(products_check.columns.tolist())
print(products_check.shape)

['product_id', 'product_category_name', 'product_category_name_english', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
(32951, 10)


In [112]:
with conn.cursor() as cur:
    with open(FINAL_DATA /"dim_products.csv","r",encoding="utf-8") as f:
        with cur.copy("""
            COPY dim_products
            FROM STDIN
            WITH (FORMAT CSV,HEADER TRUE)
        """) as copy:
            while data := f.read(1024*1024):
                copy.write(data)

conn.commit()

print("dim_products loaded")

InvalidTextRepresentation: invalid input syntax for type integer: "40.0"
CONTEXT:  COPY dim_products, line 2, column product_name_lenght: "40.0"

In [ ]:
numeric_columns = products_check.select_dtypes(include="number").columns.tolist()

print(numeric_columns)

['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [ ]:
for col in numeric_columns:
    non_integer = ((products_check[col].dropna() % 1) != 0).sum()
    print(col, "→", non_integer, "non-integer values")

product_name_lenght → 0 non-integer values
product_description_lenght → 0 non-integer values
product_photos_qty → 0 non-integer values
product_weight_g → 0 non-integer values
product_length_cm → 0 non-integer values
product_height_cm → 0 non-integer values
product_width_cm → 0 non-integer values


In [ ]:
for col in numeric_columns:
    products_check[col] = products_check[col].round().astype("Int64")

In [ ]:
print(products_check.dtypes)

product_id                         str
product_category_name              str
product_category_name_english      str
product_name_lenght              Int64
product_description_lenght       Int64
product_photos_qty               Int64
product_weight_g                 Int64
product_length_cm                Int64
product_height_cm                Int64
product_width_cm                 Int64
dtype: object


In [ ]:
POSTGRES_DATA = PROJECT_ROOT / "data" / "postgres"
POSTGRES_DATA.mkdir(exist_ok=True)

products_check.to_csv(
    POSTGRES_DATA / "dim_products.csv",
    index=False
)

In [ ]:
pd.read_csv(
    POSTGRES_DATA / "dim_products.csv"
).head()

,product_id,product_category_name,product_category_name_english,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [ ]:
integer_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_check[integer_columns] = (
    products_check[integer_columns]
    .round()
    .astype("Int64")
)

In [ ]:
products_check.dtypes

product_id                         str
product_category_name              str
product_category_name_english      str
product_name_lenght              Int64
product_description_lenght       Int64
product_photos_qty               Int64
product_weight_g                 Int64
product_length_cm                Int64
product_height_cm                Int64
product_width_cm                 Int64
dtype: object

In [ ]:
POSTGRES_DATA = PROJECT_ROOT / "data" / "postgres"
POSTGRES_DATA.mkdir(exist_ok=True)

In [ ]:
products_check.to_csv(
    POSTGRES_DATA / "dim_products.csv",
    index=False
)

In [ ]:
products_pg = pd.read_csv(
    POSTGRES_DATA / "dim_products.csv"
)

print(products_pg.head())
print(products_pg.dtypes)

                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

  product_category_name_english  product_name_lenght  \
0                     perfumery                 40.0   
1                           art                 44.0   
2                sports_leisure                 46.0   
3                          baby                 27.0   
4                    housewares                 37.0   

   product_description_lenght  product_photos_qty  product_weight_g  \
0                       287.0                 1.0             225.0   
1                       276.0                 1.0            1000.0   
2                       250.0                 1.0             154.0   
3     

In [ ]:
integer_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_check[integer_columns] = (
    products_check[integer_columns]
    .round()
    .astype("Int64")
)

# print(products_check[integer_columns].dtypes)
products_check[integer_columns].head()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,40,287,1,225,16,10,14
1,44,276,1,1000,30,18,20
2,46,250,1,154,18,9,15
3,27,261,1,371,26,4,26
4,37,402,4,625,20,17,13


In [ ]:
products_check.to_csv(
    POSTGRES_DATA / "dim_products.csv",
    index=False
)

In [ ]:
products_pg = pd.read_csv(
    POSTGRES_DATA / "dim_products.csv"
)

print(products_pg.head())

                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

  product_category_name_english  product_name_lenght  \
0                     perfumery                 40.0   
1                           art                 44.0   
2                sports_leisure                 46.0   
3                          baby                 27.0   
4                    housewares                 37.0   

   product_description_lenght  product_photos_qty  product_weight_g  \
0                       287.0                 1.0             225.0   
1                       276.0                 1.0            1000.0   
2                       250.0                 1.0             154.0   
3     